# 12 — Figure 2/3 frozen-fit diagnostics and evidence review

This is a separate validation experiment. It uses the **fixed** model families and coefficients saved by Notebook 11 and the prepared data from Notebook 08. It does not rerun PySR, refit coefficients, or redraw Monte Carlo catalogues. Notebook 11b's completed catalogue-level Monte Carlo is linked only as stability context.


## 1. What is checked?

For each frozen expression and each data subset, calculate the residual vector and its covariance-weighted score:

$$\mathbf r=\mathbf y-\mathbf f,\qquad \chi^2_S=\mathbf r_S^{\mathsf T}\mathbf C_{SS}^{-1}\mathbf r_S. $$

The subset covariance is extracted **before** inversion. The prepared DESI and SN cross-block is zero, so their chi-squared values add to the combined chi-squared. The reported $\chi^2_S/N_S$ is descriptive, not a reduced chi-squared or a p-value: both fitted branches were optimized on the joint data, so there is no unique allocation of fitted degrees of freedom to either subset.

The independent expression, the refitted exponential comparator, and the expression printed in the paper are kept distinct. The paper expression is **not** refitted. Pointwise SN residuals use marginal uncertainties and remain correlated; only the covariance-whitened residuals reproduce the full-covariance chi-squared by summing their squares.


In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd
from IPython.display import display, Image, Markdown

PROJECT_ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
                    if (p / 'results/08_fig23_data_preparation/fig2_pantheon.npz').is_file())
sys.path.insert(0, str(PROJECT_ROOT))
from src.fig23_fit_diagnostics import run_fit_diagnostics
PRIOR = (PROJECT_ROOT / 'results/11_fig23_mc_and_reconstruction' /
         'fig23_mc_20261010T044029855521Z_5cf83467')
OUTPUT_ROOT = PROJECT_ROOT / 'results/12_fig23_fit_diagnostics'
print('Frozen Notebook 11 run:', PRIOR)

## 2. Run once or inspect an existing result

The notebook opens in inspection mode. Set `RUN_ANALYSIS=True` to create one **new timestamped folder** containing both Figure 2 and Figure 3 checks, or run `scripts/notebooks/run_notebook12.py` in the project environment. Existing experiments are never overwritten.


In [ ]:
RUN_ANALYSIS = False

if RUN_ANALYSIS:
    EXPERIMENT_DIR, SUMMARY = run_fit_diagnostics(PROJECT_ROOT, PRIOR, OUTPUT_ROOT)
else:
    complete = sorted(p for p in OUTPUT_ROOT.glob('fig23_fit_check_*')
                      if (p / 'experiment.json').is_file() and
                      json.loads((p / 'experiment.json').read_text()).get('status') == 'complete')
    if not complete:
        raise FileNotFoundError('No completed Notebook 12 run; set RUN_ANALYSIS=True.')
    EXPERIMENT_DIR = complete[-1]
    SUMMARY = pd.read_csv(EXPERIMENT_DIR / 'subset_scores.csv')
print('Notebook 12 result:', EXPERIMENT_DIR)

## 3. Fit scores and residual diagnostics

Inspect DESI and SN separately before interpreting the combined loss. The highest-redshift DESI check excludes one measurement **without refitting** and is a sensitivity diagnostic, not a formal outlier test. Plotted SN marginal residuals are correlated; the whitened histogram depends on the fixed catalogue order.


In [ ]:
display(SUMMARY.round(4))
display(pd.read_csv(EXPERIMENT_DIR / 'highest_z_desi_check.csv').round(4))
display(Image(filename=str(EXPERIMENT_DIR / 'subset_score_comparison.png')))
for scenario in ('fig2_pantheon', 'fig3_desy5'):
    print(scenario)
    display(Image(filename=str(EXPERIMENT_DIR / scenario / 'residual_diagnostics.png')))

## 4. Synthesis with Notebook 11b

The table below **reads** completed Notebook 11b counts; it does not run new Monte Carlo. Fit quality, model-family sensitivity, and pole-conditioning sensitivity answer different questions. A pointwise $3\sigma$ interval at one redshift cannot by itself establish a global rejection of $\Lambda$CDM. The printed paper expression is a comparison, not the independently selected result.


In [ ]:
display(pd.read_csv(EXPERIMENT_DIR / 'notebook11b_stability_context.csv'))
display(Markdown((EXPERIMENT_DIR / 'analysis.md').read_text()))